In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import csv
import random


In [ ]:
def separate_link():
    links = []
    with open('links_tayara_csv.csv', newline='') as file:
        reader = csv.reader(file)
        header = next(reader) #skip header
        links = [row[0] for row in reader if row] #make sure rows aren't empty
    return links

In [ ]:
def scrape_ad_details(response):

    soup = BeautifulSoup(response.text, 'lxml')
    info = {}

    try:
        print(f'Page title: {soup.title.string if soup.title else 'No title found'}')

        details = soup.find('div', class_='col-span-12 pl-4 md:pl-3 pr-6 lg:pl-6 lg:col-[_span_12_/_span_9]')
        if details:

            title_div = details.find('h1')
            info['title'] = title_div.get_text(strip=True) if title_div else ''

            mt4_divs = details.find_all('div', class_='mt-4')
            for div in mt4_divs:
                price_value = div.find('data')
                if price_value and price_value.get('value'):
                    info['price'] = price_value['value']
                    break
                else:
                    info['price'] = ''

            location_div = details.find('div', class_='flex items-center space-x-2 mb-1')
            location_value = location_div.find('span')
            info['location'] = location_value.get_text(strip=True).split(',')[0] if location_value else ''
            info['publish-date'] = location_value.get_text(strip=True).split(',')[1] if location_value else ''

            #finding the description div and the specs div, since they have no classes
            total_divs = details.find_all('div')
            if not total_divs:
                print(f'Error: No divs found')
                return None
            
            description_div = None
            specs_div = None

            for div in total_divs:
                if div.find('button', class_='text-xs flex items-center gap-x-1 text-primary font-medium mt-2') and div.find('h2').get_text(strip=True) == 'Description':
                    description_div = div
                '''if div.find('ul', class_='grid gap-3 grid-cols-12') and div.find('h2').get_text(strip=True) == 'Critères':
                    specs_div = div
                    print(specs_div.prettify())'''
                h2_element = div.find('h2')
                if h2_element:
                    h2_text = h2_element.get_text(strip=True)
                    if div.find('ul', class_='grid gap-3 grid-cols-12') and h2_text == 'Critères':
                        specs_div = div

            description_value = description_div.find('p')
            info['description'] = description_value.get_text(strip=True) if description_value else ''

            '''spec_fields = [
                ('mileage', 0),
                ('model', 7),
                ('circulation-date', 4),
                ('fuel', 10),
                ('gear', 3),
                ('fiscal-power', 8),
                ('body-type', 9),
                ('brand', 6),
                ('engine-size', 10)
            ]

            #finding the specs div, since it has no class
            listed_specs = specs_div.find_all('ul', class_='grid gap-3 grid-cols-12')
            for spec, index in spec_fields:
                if index <len(listed_specs):
                    spec_element = listed_specs[index].find('span', class_='text-gray-700/80 text-xs md:text-sm lg:text-sm font-semibold')
                    info[spec] = spec_element.get_text(strip=True) if spec_element else '''''
            if specs_div:
                specs_ul = specs_div.find('ul', class_='grid gap-3 grid-cols-12')
                if specs_ul:
                    spec_items = specs_ul.find_all('li')
        
                    label_mapping = {
                        'Kilométrage': 'mileage',
                        'Modèle': 'model', 
                        'Année': 'circulation-date',
                        'Carburant': 'fuel',
                        'Boite': 'gear',
                        'Puissance fiscale': 'fiscal-power',
                        'Type de carrosserie': 'body-type',
                        'Marque': 'brand',
                        'Cylindrée': 'engine-size'
                    }
        
                    for li in spec_items:
                        label_span = li.find('span', class_='text-gray-600/80 text-2xs md:text-xs lg:text-xs font-medium pb-1 truncate')
                        value_span = li.find('span', class_='text-gray-700/80 text-xs md:text-sm lg:text-sm font-semibold')
            
                        if label_span and value_span:
                            label_text = label_span.get_text(strip=True)
                            value_text = value_span.get_text(strip=True)
                
                            # Map French label to English field name
                            if label_text in label_mapping:
                                field_name = label_mapping[label_text]
                                info[field_name] = value_text
                                #print(f"Found {field_name}: {value_text}")

            return info

        else:
            print(f"Error: Details container not found ")
            return None
    
    except Exception as e:
        print(f'Error extracting ad info: {e}')
        print(f'Details found: {details is not None if "details" in locals() else "details variable not set"}')
        print(f'Listed specs count: {len(spec_items) if "spec_items" in locals() else "spec_items not found"}')

        # Save HTML for debugging
        with open('debug_page.html', 'w', encoding='utf-8') as f:
            f.write(response.text)
        print("Saved HTML content to debug_page.html for inspection")

        return None




In [ ]:
def df_creation():
    
    links = separate_link()
    num = 1
    deleted_ads = 0
    df = pd.DataFrame(columns=[
        'title', 'price', 'brand', 'model', 'mileage', 'circulation-date',
        'fuel', 'engine-size', 'gear', 'fiscal-power', 'body-type',
        'ownership', 'publish-date', 'location', 'description', 'interior', 'link'
    ])

    successful_scrapes = 0
    failed_scrapes = 0

    for link in links:
        print(f'\n{"="*50}')
        print(f'Scraping ad number: {num}')
        print(f'URL: {link}')
        print(f'{"="*50}')

        try:
            headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'}

            response = requests.get(link, timeout = 15, headers=headers)
            response.raise_for_status()

            print(f"Response status: {response.status_code}")
            print(f"Content length: {len(response.text)} characters")

            if 'Annonce supprimée' in response.text:
                print(f" *** Skipping this ad - Deleted")
                deleted_ads += 1
                continue

            ad_details = scrape_ad_details(response)

            if ad_details:
                df = pd.concat([df, pd.DataFrame([ad_details])], ignore_index=True)
                df['link'] = link
                successful_scrapes += 1
                print(f'✓ Successfully scraped ad number: {num}')
            else:
                print(f'✗ No data extracted from ad number: {num}')

        except requests.exceptions.RequestException as e:
            print(f'✗ Failed to scrape page number: {num}; {e}')
            failed_scrapes += 1
        
        except Exception as e:
            print(f'✗ Unexpected error for ad {num}: {e}')
            failed_scrapes += 1

        num += 1
        time.sleep(random.uniform(1,2))

    total_processed = successful_scrapes + deleted_ads + failed_scrapes
    success_rate = (successful_scrapes / len(links)) * 100 if links else 0

    
    success_rate = round((successful_scrapes / len(links)) * 100, 2) if links else 0
    print(f'Success rate is: {success_rate}')
    print(f'\n{"="*50}')
    print(f'FINAL RESULTS:')
    print(f'Total links: {len(links)}')
    print(f'Successful scrapes: {successful_scrapes}')
    print(f'Success Rate: {success_rate}%')
    print(f'{"="*50}')

    df.to_csv('second_set', index = False)
    
    return df